In [ ]:
from azure.ai.inference import EmbeddingsClient
from azure.ai.inference.models import EmbeddingsOptions
from azure.core.credentials import AzureKeyCredential

client = EmbeddingsClient(
    endpoint="https://<resource>.services.ai.azure.com/models",
    credential=AzureKeyCredential("dd27a13b27ec4d8eb6f479da31951af3"),
    model="text-embedding-3-small"
)

# Create EmbeddingsOptions with inputs
options = EmbeddingsOptions(
    input=["The ultimate answer to the question of life"]
)

response = client.embed(options)
embeddings = response.data[0].embedding
print(embeddings)


In [7]:
import os
from openai import AzureOpenAI
from azure.core.credentials import AzureKeyCredential
endpoint = "https://eastus.api.cognitive.microsoft.com/"
model_name = "text-embedding-3-small"
deployment = "text-embedding-3-small"

api_version = "2024-02-01"

client = AzureOpenAI(
    api_version="2024-12-01-preview",
    azure_endpoint=endpoint,
    api_key="dd27a13b27ec4d8eb6f479da31951af3"
)

response = client.embeddings.create(
    input=["first phrase","second phrase","third phrase"],
    model=deployment
)
embeddings = []
for item in response.data:
    length = len(item.embedding)
    embeddings.append(item.embedding)
    print(
        f"data[{item.index}]: length={length}, "
        f"[{item.embedding[0]}, {item.embedding[1]}, "
        f"..., {item.embedding[length-2]}, {item.embedding[length-1]}]"
    )
print(response.usage)

data[0]: length=1536, [-0.00721184303984046, 0.007491494063287973, ..., 0.01611734740436077, -0.004887983202934265]
data[1]: length=1536, [-0.003025691257789731, 0.009231699630618095, ..., 0.029947662726044655, 0.020937401801347733]
data[2]: length=1536, [-0.013795719482004642, 0.031857650727033615, ..., 0.017506178468465805, 0.0226223636418581]
Usage(prompt_tokens=6, total_tokens=6)


In [28]:
from dotenv import load_dotenv
load_dotenv()

True

In [29]:
os.getenv("AZURE_EMBEDDING_MODEL")

'text-embedding-3-small'

In [26]:
def embed_texts(texts: list[str]) -> list[list[float]]:
    """
    Call Azure embedding endpoint for a batch of texts.
    Return list of embedding vectors.
    """
    # This depends on your Azure SDK version: if you use azure-ai-inference or azure‑ai‑openai etc.
    from azure.ai.inference import EmbeddingsClient
    from azure.core.credentials import AzureKeyCredential

    client = EmbeddingsClient("https://eastus.api.cognitive.microsoft.com/openai/deployments/text-embedding-3-small", credential=AzureKeyCredential("dd27a13b27ec4d8eb6f479da31951af3"))
    resp = client.embed(input=texts)
    # resp.data is list of embedding objects
    return [e.embedding for e in resp.data]

In [27]:
x = ["first phrase","second phrase","third phrase"]
y = embed_texts(x)

In [18]:
len(embeddings)

3

In [19]:
len(y)

3

In [22]:
y[0][0] 

-0.007211843

In [23]:
embeddings[0][0]

-0.00721184303984046

Pinecone

In [2]:
from pinecone import Pinecone, ServerlessSpec


In [3]:
pc = Pinecone()

PineconeConfigurationError: You haven't specified an API key. Please either set the PINECONE_API_KEY environment variable or pass the 'api_key' keyword argument to the Pinecone client constructor.

In [1]:
pc.list_indexes()

NameError: name 'pc' is not defined

In [42]:
pc.create_index(name="a3-index",spec=ServerlessSpec(cloud='aws', region='us-east-1'),dimension=1536)

{
    "name": "a3-index",
    "metric": "cosine",
    "host": "a3-index-x2wm50y.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 1536,
    "deletion_protection": "disabled",
    "tags": null
}

In [ ]:
def init_pinecone():
    pc = Pinecone(api_key=)
    if PINECONE_INDEX_NAME not in pc.list_indexes():
        pc.create_index(name=PINECONE_INDEX_NAME,spec=ServerlessSpec(cloud='aws', region='us-east-1'),dimension=1536)
    idx = pc.Index(PINECONE_INDEX_NAME)
    return idx

/home/zadmin/Desktop/git_repo/genai_architech/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [44]:
import json
def load_kb():
    with open("self_critique_loop_dataset.json", "r", encoding="utf-8") as f:
        items = json.load(f)
    return items

In [ ]:
def index_kb():
    kb = load_kb()
    # batch embed
    texts = [entry["question"] for entry in kb]
    embeddings = embed_texts(texts)
    idx = init_pinecone()

    # upsert in batches
    to_upsert = []
    for entry, emb in zip(kb, embeddings):
        # metadata can include id, maybe original text
        meta = {"id": entry["id"], "text": entry["text"]}
        to_upsert.append((entry["id"], emb, meta))
    # upsert
    idx.upsert(vectors=to_upsert)
    print("Indexed {} KB entries".format(len(kb)))